In [133]:
from itertools import combinations

import re
import unicodedata
import json

import pandas as pd

In [ ]:

input_dir = '/home/bsc/bsc093754/GIT/social-media-data-map/data/processed/'
input_file = f'{input_dir}participant_data_id_merged.csv'
output_dir = '/home/bsc/bsc093754/GIT/social-media-data-map/results/03_within_platform_variation'


In [135]:


def standardize_strings(df):
    """
    Standardize a string by:
    - converting to lowercase
    - removing accents
    - stripping leading/trailing whitespace
    - replacing multiple whitespace with a single space
    - removing punctuation (except letters, numbers, and spaces)
    """

    for ix, row in df.iterrows():

        text = row['keepID']

        if text is None:
            return ""

        text = str(text)

        # Lowercase
        text = text.lower()

        # Remove accents
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))

        # Remove punctuation exept colons
        text = re.sub(r"[^\w\s:]", "", text)

        # Normalize whitespace
        text = re.sub(r"\s+", " ", text).strip()

        df.at[ix, 'KeepIdStandard'] = text

    return df

 



In [136]:
def jaccard_similarity(a, b):
    set_a = set(a)
    set_b = set(b)
    # intersection of two sets
    intersection = len(set_a.intersection(set_b))
    # Unions of two sets
    union = len(set_a.union(set_b))
    
    return intersection / union

def dice_coefficient(a, b):
    """
    Compute the Dice coefficient between two collections.

    Parameters
    ----------
    a, b : iterable
        Lists, sets, tuples, etc.

    Returns
    -------
    float
        Dice coefficient in [0, 1].
    """
    set_a = set(a)
    set_b = set(b)

    if not set_a and not set_b:
        return 1.0

    intersection = len(set_a & set_b)

    return (2 * intersection) / (len(set_a) + len(set_b))



In [137]:
def group_by_participant(df):
   id_dict = df.groupby("participant")["KeepIdStandard"].apply(list).to_dict()
   #print(json.dumps(id_dict, indent = 2))
   return id_dict


In [138]:
def avg_pair_sim(input_file, output_dir, var_type, country_list):
    """
    var_type: Type of within platform variation
        1. use_var = Usage Varuation (Measure variation of DDPs due to differences in user habits)
                Replace the original JSON paths with the standardised JSON paths → Omits data design inconsistencies
                Calculate average pairwise similarity for the full DDP and stratified by topic
                Result: to what extent does the collected data in DDPs differ across users
        
        2. Data Design Variation (Measure variation of DDPs due to data design changes implemented by the platform)
                Average pairwise similarity for:
                    dd_var_consistent = a body of consistent data (username, date of joining the platform and basic demographic information)
                    dd_var_topic = data stratified by topic (only users with data on this topic included)
                Result: To what extent are DDP structures heterogenous when omitting variation due to difference is usage habits
                
    country_list: list of countries included in the analysis (eg ['ES', 'NL']
        ES = Spain
        NL = Netherlands
        LT = Lituania
        RO = Romania


    """

    df = pd.read_csv(input_file)
    df = standardize_strings(df)

    results_list = []

    for platform in df["platform"].unique():

        df_filtered = df[df["platform"] == platform]
        participant_dict = group_by_participant(df_filtered)

        output_files = {
            "use_var": "usage_var_results",
            "dd_var_consistent": "dd_var_cons_results",
            "dd_var_topic": "dd_var_topic_results",
        }

        output_file = output_files[var_type]

        country_str = '_'.join(country_list)
        output_file = f'{output_file}_{country_str}'



        n = len(participant_dict)

        if n < 2:
            print(f"{platform}: NOT ENOUGH CASES TO COMPARE")
            continue

        total = n * (n - 1) // 2

        js_total = 0
        dc_total = 0

        for (_, ids1), (_, ids2) in combinations(participant_dict.items(), 2):
            js_total += jaccard_similarity(ids1, ids2)
            dc_total += dice_coefficient(ids1, ids2)

        results_list.append({
            "platform": platform,
            "avg_js": js_total / total,
            "avg_dc": dc_total / total
        })
    results_json = json.dumps(results_list, indent=2)
    print(results_json)

    with open(f'{output_dir}/{output_file}.json', "w") as f:
        f.write(results_json)


    


avg_pair_sim(input_file, output_dir, 'use_var', ['ES', 'NL', 'LT', 'RO'])

Twitter: NOT ENOUGH CASES TO COMPARE
[
  {
    "platform": "Tiktok",
    "avg_js": 0.6363510091340937,
    "avg_dc": 0.7548749434392101
  },
  {
    "platform": "Facebook",
    "avg_js": 0.5631067961165048,
    "avg_dc": 0.7204968944099379
  },
  {
    "platform": "Instagram",
    "avg_js": 0.44810766138518643,
    "avg_dc": 0.6068304626879815
  },
  {
    "platform": "Youtube",
    "avg_js": 0.7021586412342717,
    "avg_dc": 0.805372657645385
  }
]
